# Building Races Dimension

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
 %run ../00-common/1.environment_config

In [0]:
%run ../00-common/4.gold_helpers

In [0]:
races_table = f"{catalog_name}.{silver_schema}.races"
circuits_table = f"{catalog_name}.{silver_schema}.circuits"

# target_table
target_table = f"{catalog_name}.{gold_schema}.dim_races"

In [0]:
races_df = (
    spark.table(races_table)
    .filter(F.col("batch_id") == v_batch_id)
)
circuits_df = (
    spark.table(circuits_table)
    .filter(F.col("batch_id") == v_batch_id)
)

In [0]:
dim_races_df = (
    races_df
    .join(
        circuits_df,
        races_df.circuit_id == circuits_df.circuit_id,
        "inner"
    ).select(
        races_df.season,
        races_df.round,
        races_df.race_name,
        races_df.race_date,
        circuits_df.circuit_name,
        circuits_df.locality,
        circuits_df.country
    )

)

In [0]:
display(dim_races_df)

## Writing to Gold Delta Table

In [0]:
dim_races_columns_to_update = [
    "race_name",
    "race_date",
    "circuit_name",
    "locality",
    "country"
]

write_to_gold(
    source_df=dim_races_df,
    target_table=target_table,
    merge_condition= "t.season = s.season AND t.round = s.round",
    columns_to_update = dim_races_columns_to_update
)

In [0]:
spark.table(target_table).display()